In [14]:
%pip install torch torchvision torchaudio matplotlib numpy pandas scikit-learn pillow

Note: you may need to restart the kernel to use updated packages.


In [1]:
import os
import random
import numpy as np

import torch
import torch.nn as nn
import torch.optim as optim

import matplotlib.pyplot as plt

from pathlib import Path
from torchvision import datasets, transforms
from torch.utils.data import DataLoader, random_split

In [2]:
# Set random seed
SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

# Select device
if torch.backends.mps.is_available():
    device = torch.device("mps")

elif torch.cuda.is_available():
    device = torch.device("cuda")

else:
    device = torch.device("cpu")

print("Using Device:", device)

Using Device: mps


In [4]:
# Dataset path

DATA_DIR = Path("/Users/hansamali/Desktop/ai&ml/pusl3123-ai-ml-coursework/assessment1_cifar10/data/W2 Dataset-20261104")

TRAIN_DIR = DATA_DIR / "train"
TEST_DIR = DATA_DIR / "test"

# Check folders

print("Train folder exists:", TRAIN_DIR.exists())
print("Test folder exists:", TEST_DIR.exists())

# Image transformation

transform = transforms.Compose([

    transforms.Grayscale(num_output_channels=1),

    transforms.Resize((48, 48)),

    transforms.ToTensor(),

    transforms.Normalize((0.5,), (0.5,))

])

# Load training dataset

train_dataset = datasets.ImageFolder(
    root=TRAIN_DIR,
    transform=transform
)

# Load testing dataset

test_dataset = datasets.ImageFolder(
    root=TEST_DIR,
    transform=transform
)

print("Training Images:", len(train_dataset))

print("Testing Images:", len(test_dataset))

print("Classes:", train_dataset.classes)

print("Number of Classes:", len(train_dataset.classes))

assert train_dataset.class_to_idx == test_dataset.class_to_idx, \
    "Training and testing class folders do not match!"

Train folder exists: True
Test folder exists: True
Training Images: 28709
Testing Images: 7178
Classes: ['angry', 'disgust', 'fear', 'happy', 'neutral', 'sad', 'surprise']
Number of Classes: 7


In [5]:
# Split training dataset

train_size = int(0.8 * len(train_dataset))

val_size = len(train_dataset) - train_size

train_subset, val_subset = random_split(

    train_dataset,

    [train_size, val_size],

    generator=torch.Generator().manual_seed(SEED)

)

# Training DataLoader

train_loader = DataLoader(

    train_subset,

    batch_size=64,

    shuffle=True,

    num_workers=0

)

# Validation DataLoader

val_loader = DataLoader(

    val_subset,

    batch_size=64,

    shuffle=False,

    num_workers=0

)

# Testing DataLoader

test_loader = DataLoader(

    test_dataset,

    batch_size=64,

    shuffle=False,

    num_workers=0

)

print("Training Samples:", len(train_subset))

print("Validation Samples:", len(val_subset))

print("Testing Samples:", len(test_dataset))

Training Samples: 22967
Validation Samples: 5742
Testing Samples: 7178


In [6]:
classes = train_dataset.classes

images, labels = next(iter(train_loader))

plt.figure(figsize=(12, 6))

for i in range(10):

    image = images[i].squeeze().numpy()

    image = image * 0.5 + 0.5

    plt.subplot(2, 5, i + 1)

    plt.imshow(image, cmap="gray")

    plt.title(classes[labels[i].item()])

    plt.axis("off")

plt.suptitle("Facial Emotion Dataset - Sample Images")

plt.tight_layout()

plt.show()

TimeoutError: [Errno 60] Operation timed out

In [ ]:
class CNNClassifier(nn.Module):

    def __init__(self, num_classes=7):

        super(CNNClassifier, self).__init__()

        # Convolutional Layers

        self.features = nn.Sequential(

            # Layer 1
            nn.Conv2d(1, 32, kernel_size=3, padding=1),

            nn.ReLU(),

            nn.MaxPool2d(2, 2),

            # Layer 2
            nn.Conv2d(32, 64, kernel_size=3, padding=1),

            nn.ReLU(),

            nn.MaxPool2d(2, 2),

            # Layer 3
            nn.Conv2d(64, 128, kernel_size=3, padding=1),

            nn.ReLU(),

            nn.MaxPool2d(2, 2)

        )

        # Fully Connected Layers

        self.classifier = nn.Sequential(

            nn.Flatten(),

            nn.Linear(128 * 6 * 6, 256),

            nn.ReLU(),

            nn.Linear(256, num_classes)

        )

    def forward(self, x):

        x = self.features(x)

        x = self.classifier(x)

        return x


# Create CNN Model

model = CNNClassifier(
    num_classes=len(train_dataset.classes)
).to(device)

print(model)

In [ ]:
def count_parameters(model):

    return sum(

        p.numel()

        for p in model.parameters()

        if p.requires_grad

    )


total_parameters = count_parameters(model)

print("Total Trainable Parameters in CNN:",
      total_parameters)

In [ ]:
# Loss Function

criterion = nn.CrossEntropyLoss()

# Optimizer

optimizer = optim.Adam(

    model.parameters(),

    lr=0.001

)

# Training Settings

epochs = 10

train_losses = []

val_losses = []

train_accuracies = []

val_accuracies = []

print("CNN Training Configuration Ready!")

print("Epochs:", epochs)

print("Learning Rate: 0.001")

In [ ]:
print("Starting CNN Training...")

for epoch in range(epochs):

    model.train()

    running_loss = 0.0

    correct = 0

    total = 0

    for images, labels in train_loader:

        images = images.to(device)

        labels = labels.to(device)

        # Clear gradients

        optimizer.zero_grad()

        # Forward pass

        outputs = model(images)

        # Calculate loss

        loss = criterion(outputs, labels)

        # Backpropagation

        loss.backward()

        # Update weights

        optimizer.step()

        running_loss += loss.item()

        _, predicted = torch.max(outputs, 1)

        total += labels.size(0)

        correct += (predicted == labels).sum().item()

    # Training results

    train_loss = running_loss / len(train_loader)

    train_accuracy = 100 * correct / total

    train_losses.append(train_loss)

    train_accuracies.append(train_accuracy)

    # Validation phase

    model.eval()

    val_loss = 0.0

    val_correct = 0

    val_total = 0

    with torch.no_grad():

        for images, labels in val_loader:

            images = images.to(device)

            labels = labels.to(device)

            outputs = model(images)

            loss = criterion(outputs, labels)

            val_loss += loss.item()

            _, predicted = torch.max(outputs, 1)

            val_total += labels.size(0)

            val_correct += (predicted == labels).sum().item()

    # Validation results

    val_loss = val_loss / len(val_loader)

    val_accuracy = 100 * val_correct / val_total

    val_losses.append(val_loss)

    val_accuracies.append(val_accuracy)

    print(

        f"Epoch [{epoch+1}/{epochs}] "

        f"| Train Loss: {train_loss:.4f} "

        f"| Train Acc: {train_accuracy:.2f}% "

        f"| Val Loss: {val_loss:.4f} "

        f"| Val Acc: {val_accuracy:.2f}%"

    )

print("CNN Training Completed!")

In [ ]:
model.eval()

test_correct = 0

test_total = 0

with torch.no_grad():

    for images, labels in test_loader:

        images = images.to(device)

        labels = labels.to(device)

        outputs = model(images)

        _, predicted = torch.max(outputs, 1)

        test_total += labels.size(0)

        test_correct += (predicted == labels).sum().item()


test_accuracy = 100 * test_correct / test_total

print("\nCNN Final Test Results")

print("----------------------------")

print("Total Test Images:", test_total)

print("Correct Predictions:", test_correct)

print("CNN Final Test Accuracy: {:.2f}%".format(
    test_accuracy
))

In [ ]:
plt.figure(figsize=(9, 5))

plt.plot(

    range(1, epochs + 1),

    train_losses,

    label="CNN Training Loss",

    marker="o"

)

plt.plot(

    range(1, epochs + 1),

    val_losses,

    label="CNN Validation Loss",

    marker="x"

)

plt.xlabel("Epochs")

plt.ylabel("Loss")

plt.title("Training and Validation Loss - CNN")

plt.legend()

plt.grid(True)

plt.tight_layout()

plt.show()

In [ ]:
plt.figure(figsize=(9, 5))

plt.plot(

    range(1, epochs + 1),

    train_accuracies,

    label="CNN Training Accuracy",

    marker="o"

)

plt.plot(

    range(1, epochs + 1),

    val_accuracies,

    label="CNN Validation Accuracy",

    marker="x"

)

plt.xlabel("Epochs")

plt.ylabel("Accuracy (%)")

plt.title("CNN Training and Validation Accuracy")

plt.legend()

plt.grid(True)

plt.tight_layout()

plt.show()

In [ ]:
# Create result folder

os.makedirs("CNN_Results", exist_ok=True)

# Save trained model

torch.save(

    model.state_dict(),

    "CNN_Results/cnn_emotion_model.pth"

)

print("CNN Model Saved Successfully!")